# CNN（卷积神经网络）

## Conv2d 基本用法

`nn.Conv2d(in_channels, out_channels, kernel_size)` 是 PyTorch 中的二维卷积层。

- **输入 shape**：`(batch, in_channels, H, W)`
- **输出 shape**：`(batch, out_channels, H - kernel_size + 1, W - kernel_size + 1)`（无 padding 时）
- **weight shape**：`(out_channels, in_channels, kernel_size, kernel_size)`

下面用一个随机输入验证各维度的变化：

In [1]:
import torch
in_channels, out_channels= 5, 10
width, height = 100, 100
kernel_size = 3
batch_size = 1

input = torch.randn(batch_size,
    in_channels,
    width, 
    height)

conv_layer = torch.nn.Conv2d(in_channels,
    out_channels,
    kernel_size=kernel_size)

output = conv_layer(input)

print(input.shape)
print(output.shape)
print(conv_layer.weight.shape)

torch.Size([1, 5, 100, 100])
torch.Size([1, 10, 98, 98])
torch.Size([10, 5, 3, 3])


## Padding（填充）

不加 padding 时，卷积每次会让输出比输入小 `kernel_size - 1` 个像素。设置 `padding=1` 后，会在输入四周补一圈 0，使得用 3×3 卷积核时**输出与输入尺寸保持一致**。

公式：`output_size = input_size - kernel_size + 2 * padding + 1`

下面手动设置卷积核权重，直观观察 padding 对输出的影响：

In [3]:
import torch
input = [3,4,6,5,7,
    2,4,6,8,2,
    1,6,7,8,4,
    9,7,4,6,2,
    3,7,5,4,1]
input = torch.Tensor(input).view(1, 1, 5, 5)
conv_layer = torch.nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
kernel = torch.Tensor([1,2,3,4,5,6,7,8,9]).view(1, 1, 3, 3)
conv_layer.weight.data = kernel.data
output = conv_layer(input)
print(output)

tensor([[[[ 91., 168., 224., 215., 127.],
          [114., 211., 295., 262., 149.],
          [192., 259., 282., 214., 122.],
          [194., 251., 253., 169.,  86.],
          [ 96., 112., 110.,  68.,  31.]]]], grad_fn=<ConvolutionBackward0>)


## Stride（步长）

默认 `stride=1` 时，卷积核每次移动 1 个像素。设置 `stride=2` 后，每次跳 2 个像素，**输出尺寸减半**，相当于同时做了下采样。

公式：`output_size = (input_size - kernel_size) / stride + 1`

下面用同样的输入和卷积核，对比 stride=2 时的输出：

In [4]:
import torch
input = [3,4,6,5,7,
2,4,6,8,2,
1,6,7,8,4,
9,7,4,6,2,
3,7,5,4,1]
input = torch.Tensor(input).view(1, 1, 5, 5)
conv_layer = torch.nn.Conv2d(1, 1, kernel_size=3, stride=2, bias=False)
kernel = torch.Tensor([1,2,3,4,5,6,7,8,9]).view(1, 1, 3, 3)
conv_layer.weight.data = kernel.data
output = conv_layer(input)
print(output)

tensor([[[[211., 262.],
          [251., 169.]]]], grad_fn=<ConvolutionBackward0>)


## Pooling（池化）

池化层用于**降低特征图的空间尺寸**，减少参数量，同时增加感受野。`MaxPool2d(kernel_size=2)` 在每个 2×2 区域内取最大值，输出尺寸变为原来的 1/2。

池化没有可学习参数，不会改变 channel 数量，只缩小 H 和 W。

下面演示 MaxPool2d 对 4×4 输入的效果：

In [5]:
import torch
input = [3,4,6,5,
2,4,6,8,
1,6,7,8,
9,7,4,6,
]
input = torch.Tensor(input).view(1, 1, 4, 4)
maxpooling_layer = torch.nn.MaxPool2d(kernel_size=2)
output = maxpooling_layer(input)
print(output)

tensor([[[[4., 8.],
          [9., 8.]]]])


## Implementation：用 CNN 训练 MNIST

将上面的组件组合成一个完整的 CNN 分类器，在 MNIST 手写数字数据集上训练。

**网络结构**：
```
输入 (1, 28, 28)
→ Conv2d(1→10, k=5) + ReLU + MaxPool2d(2)   → (10, 12, 12)
→ Conv2d(10→20, k=5) + ReLU + MaxPool2d(2)  → (20, 4, 4)
→ Flatten                                    → 320
→ Linear(320, 10)                            → 10类输出
```

**训练细节**：
- 损失函数：`CrossEntropyLoss`
- 优化器：`SGD`，lr=0.01，momentum=0.5
- 每 300 个 batch 打印一次平均 loss

---

### 记录最低 loss 并保存最佳模型

训练过程中 loss 不是单调下降的，最后一个 epoch 的模型不一定是最好的。标准做法是全程跟踪最低 loss，一旦出现新低就立即保存模型。

**核心代码**：
```python
best_loss = float('inf')   # 初始化为正无穷

# 在训练循环中：
if avg_loss < best_loss:
    best_loss = avg_loss
    torch.save(model.state_dict(), 'best_model.pth')
```

**`state_dict()`**：只保存模型的参数（所有层的权重和偏置），不保存模型结构，文件体积小。加载时需要先把模型结构定义好，再调用 `load_state_dict`。

**训练结束后加载最佳模型**：
```python
model.load_state_dict(torch.load('best_model.pth'))
```

注意：`load_state_dict` 要放在训练循环**结束之后**，否则文件还不存在会报 `FileNotFoundError`。

In [ ]:
import torch
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
 
# prepare dataset
 
batch_size = 64
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
 
train_dataset = datasets.MNIST(root='../dataset/mnist/', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_dataset = datasets.MNIST(root='../dataset/mnist/', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)
 
# design model using class
 
 
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = torch.nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = torch.nn.Conv2d(10, 20, kernel_size=5)
        self.pooling = torch.nn.MaxPool2d(2)
        self.fc = torch.nn.Linear(320, 10)
 
    def forward(self, x):
        batch_size = x.size(0)
        x = F.relu(self.pooling(self.conv1(x)))
        x = F.relu(self.pooling(self.conv2(x)))
        x = x.view(batch_size, -1)
        x = self.fc(x)
        return x
 
 
model = Net()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
 
# construct loss and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)
 
# training cycle forward, backward, update
best_loss = float('inf')
 
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        inputs, target = inputs.to(device), target.to(device)
        optimizer.zero_grad()
 
        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item()
        if batch_idx % 300 == 299:
            avg_loss = running_loss / 300
            print('[%d, %5d] loss: %.3f' % (epoch+1, batch_idx+1, avg_loss))
            running_loss = 0.0

            global best_loss
            if avg_loss < best_loss:
                best_loss = avg_loss
                torch.save(model.state_dict(), 'best_model.pth')
                print(f'  -> lowest loss: {best_loss:.3f}, model saved')
 
 
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print('accuracy on test set: %d %% ' % (100*correct/total))
    return correct/total
 
 
if __name__ == '__main__':
    epoch_list = []
    acc_list = []
    
    for epoch in range(10):
        train(epoch)
        acc = test()
        epoch_list.append(epoch)
        acc_list.append(acc)

    # 训练结束，加载最低 loss 的模型做最终评估
    model.load_state_dict(torch.load('best_model.pth'))
    print(f'Loaded best model (best_loss={best_loss:.3f}), Final test:')
    test()

    plt.plot(epoch_list, acc_list)
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.show()

## 技巧：用 dummy input 自动推断线性层输入大小

搭 CNN 时，`nn.Linear(?, num_classes)` 里的 `?` 取决于卷积层的输出形状，手算容易出错。

**做法**：把卷积部分单独提取出来，传入一个与真实数据同 shape 的全零张量，让 PyTorch 自动算出展平后的大小。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

conv1 = nn.Conv2d(1, 10, kernel_size=5)
conv2 = nn.Conv2d(10, 20, kernel_size=5)
pooling = nn.MaxPool2d(2)

# 构造一个与真实输入同 shape 的假输入：batch=1, channel=1, H=28, W=28
dummy = torch.zeros(1, 1, 28, 28)

# 跑一遍卷积部分（不需要训练，只是为了看 shape）
x = F.relu(pooling(conv1(dummy)))
x = F.relu(pooling(conv2(x)))       # 注意：传 x 而不是 dummy

print("展平前的 shape:", x.shape)

flat_size = x.view(1, -1).shape[1]
print("展平后的大小（即 Linear 的输入）:", flat_size)

## GoogLeNet（Inception 模块）

GoogLeNet 的核心思想是 **Inception 模块**：在同一层同时使用多种尺寸的卷积核（1×1、3×3、5×5）和池化，将结果在 channel 维度上拼接（`torch.cat`）。

这样做的好处是让网络自己学习在不同感受野中哪种更有效，而不是人为固定卷积核大小。

**InceptionA 输出 channel 数**：16（1×1分支）+ 24（5×5分支）+ 24（3×3双层分支）+ 24（池化分支）= **88**

下面实现 InceptionA 模块并将其嵌入完整网络：

In [7]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
 
# prepare dataset
 
batch_size = 64
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]) # 归一化,均值和方差
 
train_dataset = datasets.MNIST(root='../dataset/mnist/', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_dataset = datasets.MNIST(root='../dataset/mnist/', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)
 
# design model using class
class InceptionA(nn.Module):
    def __init__(self, in_channels):
        super(InceptionA, self).__init__()
        self.branch1x1 = nn.Conv2d(in_channels, 16, kernel_size=1)
 
        self.branch5x5_1 = nn.Conv2d(in_channels, 16, kernel_size=1)
        self.branch5x5_2 = nn.Conv2d(16, 24, kernel_size=5, padding=2)
 
        self.branch3x3_1 = nn.Conv2d(in_channels, 16, kernel_size=1)
        self.branch3x3_2 = nn.Conv2d(16, 24, kernel_size=3, padding=1)
        self.branch3x3_3 = nn.Conv2d(24, 24, kernel_size=3, padding=1)
 
        self.branch_pool = nn.Conv2d(in_channels, 24, kernel_size=1)
 
    def forward(self, x):
        branch1x1 = self.branch1x1(x)
 
        branch5x5 = self.branch5x5_1(x)
        branch5x5 = self.branch5x5_2(branch5x5)
 
        branch3x3 = self.branch3x3_1(x)
        branch3x3 = self.branch3x3_2(branch3x3)
        branch3x3 = self.branch3x3_3(branch3x3)
 
        branch_pool = F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)
        branch_pool = self.branch_pool(branch_pool)
 
        outputs = [branch1x1, branch5x5, branch3x3, branch_pool]
        return torch.cat(outputs, dim=1) # b,c,w,h  c对应的是dim=1
 
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(88, 20, kernel_size=5) # 88 = 24x3 + 16
 
        self.incep1 = InceptionA(in_channels=10) # 与conv1 中的10对应
        self.incep2 = InceptionA(in_channels=20) # 与conv2 中的20对应
 
        self.mp = nn.MaxPool2d(2)
        self.fc = nn.Linear(1408, 10) 
 
 
    def forward(self, x):
        in_size = x.size(0)
        x = F.relu(self.mp(self.conv1(x)))
        x = self.incep1(x)
        x = F.relu(self.mp(self.conv2(x)))
        x = self.incep2(x)
        x = x.view(in_size, -1)
        x = self.fc(x)
 
        return x
 
model = Net()
 
# construct loss and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)
 
# training cycle forward, backward, update
 
 
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        optimizer.zero_grad()
 
        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch+1, batch_idx+1, running_loss/300))
            running_loss = 0.0
 
 
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print('accuracy on test set: %d %% ' % (100*correct/total))
 
 
if __name__ == '__main__':
    for epoch in range(10):
        train(epoch)
        test()

[1,   300] loss: 0.912
[1,   600] loss: 0.198
[1,   900] loss: 0.148
accuracy on test set: 96 % 
[2,   300] loss: 0.111
[2,   600] loss: 0.106
[2,   900] loss: 0.091
accuracy on test set: 97 % 
[3,   300] loss: 0.079
[3,   600] loss: 0.079
[3,   900] loss: 0.071
accuracy on test set: 98 % 
[4,   300] loss: 0.063
[4,   600] loss: 0.066
[4,   900] loss: 0.062
accuracy on test set: 98 % 
[5,   300] loss: 0.056
[5,   600] loss: 0.052
[5,   900] loss: 0.055
accuracy on test set: 98 % 
[6,   300] loss: 0.048
[6,   600] loss: 0.048
[6,   900] loss: 0.053
accuracy on test set: 98 % 
[7,   300] loss: 0.042
[7,   600] loss: 0.047
[7,   900] loss: 0.042
accuracy on test set: 98 % 
[8,   300] loss: 0.042
[8,   600] loss: 0.041
[8,   900] loss: 0.040
accuracy on test set: 98 % 
[9,   300] loss: 0.035
[9,   600] loss: 0.039
[9,   900] loss: 0.037
accuracy on test set: 98 % 
[10,   300] loss: 0.033
[10,   600] loss: 0.032
[10,   900] loss: 0.038
accuracy on test set: 98 % 


## ResNet（残差网络）

深层网络训练时容易出现**梯度消失**问题，导致越深反而越难训练。ResNet 通过**跳跃连接（skip connection）**解决这个问题：

```
输出 = F(x) + x
```

即在两层卷积的基础上，直接把输入 `x` 加到输出上。这样即使卷积部分什么都没学到（权重为 0），梯度也能通过 `x` 直接反传，不会消失。

`ResidualBlock` 要求输入输出的 channel 数相同，以便做加法。

下面实现 ResidualBlock 并将其嵌入完整网络：

In [8]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
 
# prepare dataset
 
batch_size = 64
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]) # 归一化,均值和方差
 
train_dataset = datasets.MNIST(root='../dataset/mnist/', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_dataset = datasets.MNIST(root='../dataset/mnist/', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)
 
# design model using class
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.channels = channels
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
 
    def forward(self, x):
        y = F.relu(self.conv1(x))
        y = self.conv2(y)
        return F.relu(x + y)
 
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5) # 88 = 24x3 + 16
 
        self.rblock1 = ResidualBlock(16)
        self.rblock2 = ResidualBlock(32)
 
        self.mp = nn.MaxPool2d(2)
        self.fc = nn.Linear(512, 10) # 暂时不知道1408咋能自动出来的
 
 
    def forward(self, x):
        in_size = x.size(0)
 
        x = self.mp(F.relu(self.conv1(x)))
        x = self.rblock1(x)
        x = self.mp(F.relu(self.conv2(x)))
        x = self.rblock2(x)
 
        x = x.view(in_size, -1)
        x = self.fc(x)
        return x
 
model = Net()
 
# construct loss and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)
 
# training cycle forward, backward, update
 
 
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        optimizer.zero_grad()
 
        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch+1, batch_idx+1, running_loss/300))
            running_loss = 0.0
 
 
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print('accuracy on test set: %d %% ' % (100*correct/total))
 
 
if __name__ == '__main__':
    for epoch in range(10):
        train(epoch)
        test()

[1,   300] loss: 0.573
[1,   600] loss: 0.153
[1,   900] loss: 0.116
accuracy on test set: 97 % 
[2,   300] loss: 0.084
[2,   600] loss: 0.083
[2,   900] loss: 0.078
accuracy on test set: 97 % 
[3,   300] loss: 0.060
[3,   600] loss: 0.061
[3,   900] loss: 0.059
accuracy on test set: 98 % 
[4,   300] loss: 0.049
[4,   600] loss: 0.047
[4,   900] loss: 0.050
accuracy on test set: 98 % 
[5,   300] loss: 0.040
[5,   600] loss: 0.041
[5,   900] loss: 0.044
accuracy on test set: 98 % 
[6,   300] loss: 0.035
[6,   600] loss: 0.036
[6,   900] loss: 0.039
accuracy on test set: 98 % 
[7,   300] loss: 0.033
[7,   600] loss: 0.030
[7,   900] loss: 0.035
accuracy on test set: 98 % 
[8,   300] loss: 0.026
[8,   600] loss: 0.033
[8,   900] loss: 0.027
accuracy on test set: 98 % 
[9,   300] loss: 0.022
[9,   600] loss: 0.027
[9,   900] loss: 0.026
accuracy on test set: 98 % 
[10,   300] loss: 0.022
[10,   600] loss: 0.023
[10,   900] loss: 0.025
accuracy on test set: 98 % 
